## Оценка качества извлечения

Идея бенчмарка состоит в измерении качества извлечения документов по описанию (docstring) типов, методов, функций и т.п., содержащихся в них, в зависимости от используемого алгортима сегментации исходных текстов. Сами docstring при этом удаляются из исходного кода. Помимо представленного алгоритма на основе AST, используются стандартные функции сегментации из LlamaIndex:

- `SentenceSplitter` - базовый алгоритм разбиения текстов на фиксированные сегменты.
- `CodeSplitter` - алгоритм, разбивающий код только по границам блоков кода.
- `BM25` - алгоритм Okapi BM25 (без сегментации, поиск осуществляется по всему документу).

In [ ]:
import time
import torch
import datasets
import pandas as pd
import numpy as np

from qdrant_client import QdrantClient
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, Settings, StorageContext, QueryBundle
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from tqdm.std import tqdm
from beir.retrieval.evaluation import EvaluateRetrieval
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine.retriever_query_engine import RetrieverQueryEngine
from llama_index.core.node_parser import CodeSplitter
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.retrievers.fusion_retriever import FUSION_MODES
from llama_index.core.postprocessor import SentenceTransformerRerank
from chunking import SourceCodeNodeParser
from openai import Client
from pydantic import BaseModel


np.random.seed(42)

class Relevance(BaseModel):
    rate: int

Для оценки качества извлечения данных был собран [набор данных](https://huggingface.co/datasets/kmvi/code-ir-dataset), состоящий из трех блоков:

- `docs` - файлы исходного кода (документы) с удаленными из них docstring:
    - `doc_id` - идентификатор документа
    - `file` - имя файла
    - `content` - содержимое файла (с удаленными docstring's)
- `queries` - запросы к документам. В качестве запросов выступают docstring
    - `query_id` - идентификатор запроса
    - `query` - текст запроса
    - `type` - тип элемента кода, к которому относился docstring (тип, либо метод/функция)
- `qrels` - корректные пары запрос/документ
    - `query_id` - идентификатор запроса
    - `doc_id` - идентификатор документа

In [2]:
docs_ds = datasets.load_dataset("kmvi/code-ir-dataset", name="docs")
queries_ds = datasets.load_dataset("kmvi/code-ir-dataset", name="queries")
qrels_ds = datasets.load_dataset("kmvi/code-ir-dataset", name="qrels")

doc_ids = { row['file']: row['doc_id'] for row in docs_ds['train'] }
qrels = qrels_ds['train'].to_pandas().groupby('query_id').doc_id.apply(list).apply(lambda x: {k: 1 for k in x})

Подготавливаем исходные тексты для построения индекса.

In [3]:
docs = []
for row in docs_ds['train']:
    with open(row['file'], 'rt', encoding='utf-8') as f:
        text = f.read().replace('\r\n', '\n')

    doc = Document(id_=row['doc_id'], text=text, metadata={'file_name': row['file']})
    docs.append(doc)

Загрузка модели эмбеддингов:

In [ ]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    device="cuda",
    normalize=True,
    model_kwargs={'dtype': torch.bfloat16},
)

Settings.llm = None

LLM is explicitly disabled. Using MockLLM.


Построение индекса с использованием SentenceSplitter.

In [ ]:
client = QdrantClient(url='http://192.168.122.219:6333')
vector_store = QdrantVectorStore(client=client, collection_name='ir_base')
storage_context = StorageContext.from_defaults(vector_store=vector_store)
splitter = SentenceSplitter(chunk_size=256)
index = VectorStoreIndex.from_documents(docs, storage_context, transformations=[splitter], show_progress=True, store_nodes_override=True)
client.close()

Parsing nodes:   0%|          | 0/175 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/673 [00:00<?, ?it/s]

Построение индекса с использованием CodeSplitter:

In [ ]:
client = QdrantClient(url='http://192.168.122.219:6333')
vector_store = QdrantVectorStore(client=client, collection_name='ir_codesplitter')
storage_context = StorageContext.from_defaults(vector_store=vector_store)
splitter = CodeSplitter('csharp', max_chars=256)
index = VectorStoreIndex.from_documents(docs, storage_context, transformations=[splitter], show_progress=True, store_nodes_override=True)
client.close()

Parsing nodes:   0%|          | 0/175 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/276 [00:00<?, ?it/s]

Построение индекса с использованием нового алгоритма на основе AST:

In [ ]:
client = QdrantClient(url='http://192.168.122.219:6333')
vector_store = QdrantVectorStore(client=client, collection_name='ir_astsplitter')
storage_context = StorageContext.from_defaults(vector_store=vector_store)
parser = SourceCodeNodeParser(chunk_size=256)
nodes = parser(docs)
index = VectorStoreIndex(nodes, storage_context=storage_context, show_progress=True, store_nodes_override=True)
client.close()

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/969 [00:00<?, ?it/s]

## Вычисление метрик

Для расчета метрик используется функционал из библиотеки `beir`.

In [ ]:
retr = EvaluateRetrieval()
top_k = 10 # извлекаем 10 документов
client = QdrantClient(url='http://192.168.122.219:6333')

In [ ]:
storename = 'ir_base'
vector_store = QdrantVectorStore(client=client, collection_name=storename)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)

Функция для расчета метрик. Также возвращает пары "запрос - извлеченные фрагменты", которые будут использоваться для оценки релевантности.

In [ ]:
def run_benchmark(engine):
    tm = time.perf_counter()
    results = dict()
    scores, retr_data = [], []
    for row in tqdm(queries_ds['train']):
        data = engine.retrieve(QueryBundle(row['query']))
        retr_data.append((row['query'], [node.text for node in data]))
        results[row['query_id']] = { doc_ids[node.metadata['file_name']]: node.score for node in data}
        scores.append(np.mean([node.score for node in data]))
    
    tm = time.perf_counter() - tm
    tm /= len(queries_ds['train'])

    metrics = retr.evaluate(qrels, results, k_values=[3,5,10])
    
    results = dict()
    for item in (*metrics, {'Latency': tm * 1000, 'Mean Score': np.mean(scores)}):
        results.update(item)

    return results, retr_data

In [ ]:
engine = index.as_query_engine(similarity_top_k=top_k)
metrics, retr_data = run_benchmark(engine)

100%|██████████| 1030/1030 [06:02<00:00,  2.85it/s]


In [ ]:
storename = 'ir_codesplitter'
vector_store = QdrantVectorStore(client=client, collection_name=storename)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)

In [ ]:
engine = index.as_query_engine(similarity_top_k=top_k)
metrics_codesplit, retr_data_codesplit = run_benchmark(engine)

100%|██████████| 1030/1030 [08:03<00:00,  2.13it/s]


In [ ]:
storename = 'ir_astsplitter'
vector_store = QdrantVectorStore(client=client, collection_name=storename)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)

In [ ]:
engine = index.as_query_engine(similarity_top_k=top_k)
metrics_ast, retr_data_ast = run_benchmark(engine)

100%|██████████| 1030/1030 [09:35<00:00,  1.79it/s]


In [ ]:
tm = time.perf_counter()
docids = list(doc_ids.values())
results_random = dict()
retr_data_random = []
for row in tqdm(queries_ds['train']):
    idx = np.random.choice(range(0, len(nodes)), top_k)
    results_random[row['query_id']] = { doc_ids[node.metadata['file_name']]: 1.0 / top_k for node in nodes }
    retr_data_random.append((row['query'], [node.text for node in nodes]))

tm = time.perf_counter() - tm
tm /= len(queries_ds['train'])

metrics_random = retr.evaluate(qrels, results_random, k_values=[3,5,10])
metrics_random = (*metrics_random, {'Latency': tm * 1000, 'Mean Score': -1})

results_random = dict()
for item in metrics_random:
    results_random.update(item)

metrics_random = results_random

100%|██████████| 1030/1030 [00:00<00:00, 5916.78it/s]


In [ ]:
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k)
engine = RetrieverQueryEngine.from_args(bm25, llm=Settings.llm)
metrics_bm25, retr_data_bm25 = run_benchmark(engine)

100%|██████████| 1030/1030 [00:01<00:00, 866.67it/s]


Дополнительно расчитаем релевантность извлеченных фрагментов исходному запросу с помощью LLM. Попросим LLM оценить релевантность по шкале от 0 до 10. Чтобы упростить получение ответа, используем структурный вывод. В качестве оценивающей LLM использовалась модель [Google Gemma 4](https://huggingface.co/google/gemma-4-26B-A4B-it).

In [ ]:
def calc_relevance(retr_data):
    prompt_template = """You need to rate the relevance of the function or class description to its source code on a scale of 0 to 10.
    Description:
    {0}
    Source code:
    {1}
    """

    rates = []
    client = Client(base_url='http://localhost:8080', api_key='')

    pg = tqdm(retr_data)
    for query, chunks in pg:
        values = []
        for chunk in chunks[:1]:
            prompt = prompt_template.format(query, chunk)
            result = client.chat.completions.parse(
                model='',
                messages=[{'role': 'user', 'content': prompt}],
                response_format=Relevance,
                temperature=0, top_p=1, seed=42,
            )
            msg = result.choices[0].message
            if not msg.refusal:
                values.append(msg.parsed.rate)
        rates.append(np.mean(values))
        pg.set_postfix_str(np.mean(rates))

    client.close()

    return np.mean(rates)

In [ ]:
rate_base = calc_relevance(retr_data)

100%|██████████| 1030/1030 [32:22<00:00,  1.89s/it, 6.50873786407767]  


In [ ]:
rate_codesplit = calc_relevance(retr_data_codesplit)

100%|██████████| 1030/1030 [25:36<00:00,  1.49s/it, 7.245631067961165] 


In [ ]:
rate_ast = calc_relevance(retr_data_ast)

100%|██████████| 1030/1030 [25:16<00:00,  1.47s/it, 7.371844660194175] 


In [ ]:
rate_bm25 = calc_relevance(retr_data_bm25)

100%|██████████| 1030/1030 [26:00<00:00,  1.51s/it, 2.9737864077669904]


In [ ]:
rate_random = calc_relevance(retr_data_random)

100%|██████████| 1030/1030 [24:19<00:00,  1.42s/it, 0.13592233009708737]


In [ ]:
metrics.update({'Relevance': rate_base / 10})
metrics_codesplit.update({'Relevance': rate_codesplit / 10})
metrics_ast.update({'Relevance': rate_ast / 10})
metrics_random.update({'Relevance': rate_random / 10})
metrics_bm25.update({'Relevance': rate_bm25 / 10})

Итоговый результат:

In [ ]:
rows = [metrics_random, metrics_bm25, metrics, metrics_codesplit, metrics_ast]
df = pd.DataFrame(rows, index=['Random', 'BM25', 'SentenceSplitter', 'CodeSplitter', 'AST Splitter'])
df = df[df.filter(regex='^NDCG|^Recall|^P|^La|^Me|^Re').columns]

display(df.style.highlight_max(color='green').format(precision=4))

,NDCG@3,NDCG@5,NDCG@10,Recall@3,Recall@5,Recall@10,P@3,P@5,P@10,Latency,Mean Score,Relevance
Random,0.0104,0.0136,0.0213,0.0132,0.0206,0.0444,0.0052,0.0052,0.0052,0.1558,-1.0000,0.0136
BM25,0.2564,0.2933,0.3335,0.3026,0.3907,0.5123,0.1097,0.0854,0.0567,1.2312,7.0386,0.2974
SentenceSplitter,0.5996,0.6172,0.6193,0.6884,0.7281,0.7339,0.2450,0.1573,0.0792,354.3026,0.5694,0.6509
CodeSplitter,0.5889,0.6265,0.6291,0.7203,0.8080,0.8153,0.2573,0.1742,0.0880,474.3069,0.6027,0.7246
AST Splitter,0.6072,0.6360,0.6413,0.7380,0.8056,0.8206,0.2644,0.1740,0.0885,560.6683,0.6123,0.7372


### Дополнительные подходы

Попробуем улучшить качество поиска с помощью следующих подходов:
- Совмещение результатов алгоритмов AST Splitter и BM25 (гибридный поиск)
- Применение реранжирования с помощью различных моделей: BAAI/bge-reranker-v2-m3, 
- Применение реранжирования и гибридного подхода

In [ ]:
storename = 'ir_astsplitter'
vector_store = QdrantVectorStore(client=client, collection_name=storename)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)

In [ ]:
parser = SourceCodeNodeParser(chunk_size=256)
nodes = parser(docs)
retriever = index.as_retriever(similarity_top_k=top_k+5)
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k+5)
retriever = QueryFusionRetriever(
    retrievers=[retriever, bm25],
    similarity_top_k=top_k,
    use_async=False, verbose=False, mode=FUSION_MODES.RECIPROCAL_RANK, num_queries=1)

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm)

metrics_hybrid, retr_data_hybrid = run_benchmark(engine)
metrics_hybrid

100%|██████████| 1030/1030 [00:36<00:00, 28.37it/s]


{'NDCG@3': 0.34113,
 'NDCG@5': 0.44963,
 'NDCG@10': 0.4759,
 'MAP@3': 0.28965,
 'MAP@5': 0.35121,
 'MAP@10': 0.36388,
 'Recall@3': 0.47288,
 'Recall@5': 0.73553,
 'Recall@10': 0.80932,
 'P@3': 0.17184,
 'P@5': 0.15864,
 'P@10': 0.08738,
 'MRR@3': 0.31893,
 'MRR@5': 0.38083,
 'MRR@10': 0.38818,
 'Latency': 35.2551144660785,
 'Mean Score': 0.01820017350711033}

In [ ]:
retriever = index.as_retriever(similarity_top_k=top_k)
reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-v2-m3", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank_bge, retr_data_rerank_bge = run_benchmark(engine)
metrics_rerank_bge

100%|██████████| 1030/1030 [02:35<00:00,  6.64it/s]


{'NDCG@3': 0.61032,
 'NDCG@5': 0.64609,
 'NDCG@10': 0.65074,
 'MAP@3': 0.56603,
 'MAP@5': 0.58789,
 'MAP@10': 0.59009,
 'Recall@3': 0.72385,
 'Recall@5': 0.80754,
 'Recall@10': 0.82065,
 'P@3': 0.25955,
 'P@5': 0.17437,
 'P@10': 0.08854,
 'MRR@3': 0.58026,
 'MRR@5': 0.59846,
 'MRR@10': 0.6005,
 'Latency': 150.59456504854256,
 'Mean Score': 0.2172928}

In [ ]:
parser = SourceCodeNodeParser(chunk_size=256)
nodes = parser(docs)

retriever = index.as_retriever(similarity_top_k=top_k+5)
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k+5)
retriever2 = bm25
retriever = QueryFusionRetriever(
    retrievers=[retriever, retriever2],
    similarity_top_k=top_k+5,
    use_async=False, verbose=False, mode=FUSION_MODES.RECIPROCAL_RANK, num_queries=1)
reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-v2-m3", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank25_bge, retr_data_rerank25_bge = run_benchmark(engine)
metrics_rerank25_bge

100%|██████████| 1030/1030 [04:36<00:00,  3.72it/s]


{'NDCG@3': 0.53665,
 'NDCG@5': 0.58502,
 'NDCG@10': 0.59792,
 'MAP@3': 0.48431,
 'MAP@5': 0.51276,
 'MAP@10': 0.51891,
 'Recall@3': 0.6745,
 'Recall@5': 0.78816,
 'Recall@10': 0.8244,
 'P@3': 0.24337,
 'P@5': 0.17068,
 'P@10': 0.08942,
 'MRR@3': 0.49757,
 'MRR@5': 0.52413,
 'MRR@10': 0.5296,
 'Latency': 268.49347320407884,
 'Mean Score': 0.22749756}

In [ ]:
retriever = index.as_retriever(similarity_top_k=top_k)
reranker = SentenceTransformerRerank(model="cross-encoder/stsb-roberta-large", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank_roberta, retr_data_rerank_roberta = run_benchmark(engine)
metrics_rerank_roberta

100%|██████████| 1030/1030 [03:08<00:00,  5.48it/s]


{'NDCG@3': 0.59618,
 'NDCG@5': 0.62829,
 'NDCG@10': 0.63154,
 'MAP@3': 0.54343,
 'MAP@5': 0.56311,
 'MAP@10': 0.56471,
 'Recall@3': 0.73647,
 'Recall@5': 0.81159,
 'Recall@10': 0.82065,
 'P@3': 0.26343,
 'P@5': 0.17515,
 'P@10': 0.08854,
 'MRR@3': 0.55437,
 'MRR@5': 0.57058,
 'MRR@10': 0.57202,
 'Latency': 182.55778631055276,
 'Mean Score': 0.45540592}

In [ ]:
retriever = index.as_retriever(similarity_top_k=top_k+5)
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k+5)
retriever2 = bm25
retriever = QueryFusionRetriever(
    retrievers=[retriever, retriever2],
    similarity_top_k=top_k+5,
    use_async=False, verbose=False, mode=FUSION_MODES.RECIPROCAL_RANK, num_queries=1)
reranker = SentenceTransformerRerank(model="cross-encoder/stsb-roberta-large", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank25_bert, retr_data_rerank25_bert = run_benchmark(engine)
metrics_rerank25_bert

DEBUG:bm25s:Building index from IDs objects
100%|██████████| 1030/1030 [05:51<00:00,  2.93it/s]


{'NDCG@3': 0.32828,
 'NDCG@5': 0.43895,
 'NDCG@10': 0.46654,
 'MAP@3': 0.27297,
 'MAP@5': 0.33546,
 'MAP@10': 0.34899,
 'Recall@3': 0.4789,
 'Recall@5': 0.74479,
 'Recall@10': 0.82133,
 'P@3': 0.1712,
 'P@5': 0.15942,
 'P@10': 0.08874,
 'MRR@3': 0.28204,
 'MRR@5': 0.34257,
 'MRR@10': 0.35376,
 'Latency': 341.233354562993,
 'Mean Score': 0.49069837}

In [ ]:
retriever = index.as_retriever(similarity_top_k=top_k)
reranker = SentenceTransformerRerank(model="Qwen/Qwen3-Reranker-0.6B", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank_qwen, retr_data_qwen = run_benchmark(engine)
metrics_rerank_qwen

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

c:\Users\user\miniconda3\envs\ml_main\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--Qwen--Qwen3-Reranker-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/741 [00:00<?, ?B/s]

  0%|          | 0/1030 [00:00<?, ?it/s]You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
c:\Users\user\miniconda3\envs\ml_main\Lib\site-packages\transformers\tokenization_utils_base.py:2914: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
100%|██████████| 1030/1030 [03:23<00:00,  5.06it/s]


{'NDCG@3': 0.51498,
 'NDCG@5': 0.56297,
 'NDCG@10': 0.57419,
 'MAP@3': 0.45565,
 'MAP@5': 0.48417,
 'MAP@10': 0.48978,
 'Recall@3': 0.67725,
 'Recall@5': 0.78958,
 'Recall@10': 0.82065,
 'P@3': 0.24078,
 'P@5': 0.1701,
 'P@10': 0.08854,
 'MRR@3': 0.46343,
 'MRR@5': 0.48877,
 'MRR@10': 0.493,
 'Latency': 197.76748368941537,
 'Mean Score': 0.006716057911944657}

In [ ]:
retriever = index.as_retriever(similarity_top_k=top_k+5)
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k+5)
retriever2 = bm25
retriever = QueryFusionRetriever(
    retrievers=[retriever, retriever2],
    similarity_top_k=top_k+5,
    use_async=False, verbose=False, mode=FUSION_MODES.RECIPROCAL_RANK, num_queries=1)
reranker = SentenceTransformerRerank(model="Qwen/Qwen3-Reranker-0.6B", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank25_qwen, retr_data_rerank25_qwen = run_benchmark(engine)
metrics_rerank25_qwen

DEBUG:bm25s:Building index from IDs objects
Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-Reranker-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 1030/1030 [06:07<00:00,  2.80it/s]


{'NDCG@3': 0.26598,
 'NDCG@5': 0.38032,
 'NDCG@10': 0.42066,
 'MAP@3': 0.21776,
 'MAP@5': 0.28243,
 'MAP@10': 0.30126,
 'Recall@3': 0.39408,
 'Recall@5': 0.66696,
 'Recall@10': 0.78233,
 'P@3': 0.14272,
 'P@5': 0.14427,
 'P@10': 0.08466,
 'MRR@3': 0.22913,
 'MRR@5': 0.29262,
 'MRR@10': 0.30947,
 'Latency': 356.74607621356665,
 'Mean Score': 0.94873846}

Оценка релевантности поиска с помощью LLM:

In [ ]:
rate_hybrid = calc_relevance(retr_data_hybrid)

100%|██████████| 1030/1030 [25:49<00:00,  1.50s/it, 6.680582524271845] 


In [ ]:
rate_rerank25 = calc_relevance(retr_data_rerank25_bge)

100%|██████████| 1030/1030 [26:27<00:00,  1.54s/it, 7.01747572815534]  


In [ ]:
rate_rerank = calc_relevance(retr_data_rerank_bge)

100%|██████████| 1030/1030 [26:13<00:00,  1.53s/it, 7.575728155339806] 


In [ ]:
rate_qwen = calc_relevance(retr_data_qwen)

100%|██████████| 1030/1030 [24:33<00:00,  1.43s/it, 5.209708737864077] 


In [ ]:
rate_roberta = calc_relevance(retr_data_rerank_roberta)

100%|██████████| 1030/1030 [25:15<00:00,  1.47s/it, 6.450485436893204] 


In [ ]:
rate_roberta25 = calc_relevance(retr_data_rerank25_bert)

100%|██████████| 1030/1030 [26:28<00:00,  1.54s/it, 4.295145631067961] 


In [ ]:
rate_qwen25 = calc_relevance(retr_data_rerank25_qwen)

100%|██████████| 1030/1030 [25:17<00:00,  1.47s/it, 3.8281553398058255]


In [ ]:
metrics_hybrid.update({'Relevance': rate_hybrid / 10})
metrics_rerank_bge.update({'Relevance': rate_rerank / 10})
metrics_rerank25_bge.update({'Relevance': rate_rerank25 / 10})
metrics_rerank_qwen.update({'Relevance': rate_qwen / 10})
metrics_rerank_roberta.update({'Relevance': rate_roberta / 10})
metrics_rerank25_bert.update({'Relevance': rate_roberta25 / 10})
metrics_rerank25_qwen.update({'Relevance': rate_qwen25 / 10})

Итоговый результат:

In [ ]:
rows = [metrics_hybrid, metrics_rerank25_bge, metrics_rerank_roberta, metrics_rerank_qwen, metrics_rerank_bge, metrics_rerank25_bert, metrics_rerank25_qwen]

df = pd.DataFrame(rows, index=['AST Splitter + BM25', 'AST Splitter + BM25 + rerank BGE',
                               'AST Splitter + rerank RoBERTa', 'AST Splitter + rerank Qwen', 'AST Splitter + rerank BGE',
                               'AST Splitter + BM25 + rerank RoBERTa', 'AST Splitter + BM25 + rerank Qwen'])
df = df[df.filter(regex='^NDCG|^Recall|^P|^La|^Me|^Re').columns]

display(df.style.highlight_max(color='green').format(precision=4))

,NDCG@3,NDCG@5,NDCG@10,Recall@3,Recall@5,Recall@10,P@3,P@5,P@10,Latency,Mean Score,Relevance
AST Splitter + BM25,0.3411,0.4496,0.4759,0.4729,0.7355,0.8093,0.1718,0.1586,0.0874,33.1137,0.0182,0.6681
AST Splitter + BM25 + rerank BGE,0.5366,0.5850,0.5979,0.6745,0.7882,0.8244,0.2434,0.1707,0.0894,268.4935,0.2275,0.7017
AST Splitter + rerank RoBERTa,0.5962,0.6283,0.6315,0.7365,0.8116,0.8206,0.2634,0.1752,0.0885,178.6905,0.4554,0.6450
AST Splitter + rerank Qwen,0.5343,0.5777,0.5873,0.6909,0.7940,0.8206,0.2476,0.1711,0.0885,187.8500,0.4115,0.5210
AST Splitter + rerank BGE,0.6103,0.6461,0.6507,0.7238,0.8075,0.8206,0.2596,0.1744,0.0885,150.5946,0.2173,0.7576
AST Splitter + BM25 + rerank RoBERTa,0.3283,0.4390,0.4665,0.4789,0.7448,0.8213,0.1712,0.1594,0.0887,338.0665,0.4907,0.4295
AST Splitter + BM25 + rerank Qwen,0.2767,0.3939,0.4308,0.4039,0.6857,0.7902,0.1463,0.1484,0.0855,353.2382,0.4149,0.3828
